[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Giocrisrai/mly1101-machine-learning/blob/main/notebooks/11_alumno_seleccion.ipynb)

# MLY1101 · Machine Learning — Actividad 3.3
## Robustez y selección de modelos

**Resultado de aprendizaje (RA3):** elabora soluciones avanzadas de aprendizaje automático
mediante la optimización de hiperparámetros, técnicas de ensamble y validación cruzada, para
garantizar la precisión y generalización del modelo frente a objetivos de negocio complejos.

**Indicadores de logro:**

- **IL 3.3** · evalúa la capacidad de generalización del modelo mediante esquemas de validación
  cruzada y métricas de desempeño avanzadas según la naturaleza del problema.
- **IL 3.4** · sustenta la selección de la solución analítica óptima mediante la comparación
  cuantitativa de modelos, asegurando el cumplimiento de los objetivos del negocio.

---

### La pregunta que cierra la asignatura

Llevas tres actividades midiendo modelos. Ninguna ha respondido lo único que importa al final:

> **¿Cuál eliges, y cómo lo defiendes?**

No es una pregunta de estadística. La estadística te dice si dos modelos son distinguibles; la
elección la haces tú, con criterios que la métrica no contiene.

---

### La idea central, y es incómoda

> **Casi todas las comparaciones de modelos que verás en internet no distinguen nada.**

Alguien reporta 0,847 contra 0,843 y concluye que el primero es mejor. Si la variabilidad entre
particiones es 0,012, esa diferencia es ruido: con otra semilla se habría invertido el orden.

Hoy vas a aprender a hacer esa comparación bien. Y vas a descubrir que el modelo que llevas
usando desde la Actividad 2.2 **no es distinguible** del ensamble que construiste ayer.

---

### Al final de la sesión debes entregar

La **tabla de selección sustentada**: métrica, estabilidad, costo e interpretabilidad juntos,
con la decisión argumentada. Es el esqueleto del apartado de modelamiento del EFT.

---
## Preparación del entorno

In [ ]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO = Path("mly1101-machine-learning")
    if not REPO.exists():
        !git clone -q https://github.com/Giocrisrai/mly1101-machine-learning.git {REPO}
    RAIZ = REPO.resolve()
else:
    RAIZ = Path("..").resolve()

sys.path.insert(0, str(RAIZ / "src"))
sys.path.insert(0, str(RAIZ / "kedro_mly1101" / "src"))

RUTA_DATOS = RAIZ / "datos" / "crudos" / "detecciones_waymo_like.csv"
RUTA_PARAMETROS = RAIZ / "kedro_mly1101" / "conf" / "base" / "parameters.yml"
print("Colab:", EN_COLAB, "| dataset:", RUTA_DATOS.exists())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from kedro_mly1101.pipelines.preprocesamiento import nodes as limpieza
from kedro_mly1101.pipelines.supervisado import nodes as supervisado
from kedro_mly1101.pipelines.optimizacion import nodes as optimizacion

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 150)
sns.set_theme(style="whitegrid")

PARAMETROS = yaml.safe_load(RUTA_PARAMETROS.read_text(encoding="utf-8"))
CONFIG, FUGA, AJUSTE = PARAMETROS["modelo"], PARAMETROS["fuga"], PARAMETROS["ajuste"]

# La misma cadena de siempre: limpieza del RA1 -> partición del RA2.
crudo = pd.read_csv(RUTA_DATOS)
paso = limpieza.normalizar_categorias(crudo, PARAMETROS["mapas_categorias"])
paso = limpieza.descubrir_faltantes(paso, PARAMETROS["centinelas"])
paso = limpieza.marcar_imposibles(paso, PARAMETROS["reglas_dominio"])
limpio = limpieza.quitar_duplicados_y_constantes(paso, PARAMETROS["columnas_a_descartar"])

marcada = supervisado.particionar(
    supervisado.preparar_variables(limpio, CONFIG, FUGA), CONFIG
)
entrena = marcada[marcada["particion"] == "entrenamiento"]

print(f"Entrenamiento: {len(entrena):,} filas en {entrena[CONFIG['grupo']].nunique()} segmentos")
print(f"Métrica de trabajo: {AJUSTE['metrica']}  ·  pliegues: {AJUSTE['n_pliegues']}")

---
# Bloque 1 · ⭐⭐ ¿Qué diferencias se pueden distinguir?

Cada pliegue de la validación cruzada da una puntuación distinta, porque le tocan segmentos
distintos. Esa dispersión **no es un defecto de la medición**: es la incertidumbre real de tu
estimación.

Y da la regla práctica: **si dos modelos difieren menos que esa dispersión, no puedes afirmar
que uno sea mejor.**

### ✏️ TODO 1 — Ver la dispersión, no solo la media

In [ ]:
# TODO 1: compara los modelos mostrando la dispersión, no solo la media.
comparacion = optimizacion.____(marcada, CONFIG, AJUSTE)
print(comparacion.to_string(index=False), "\n")

fig, ejes = plt.subplots(figsize=(7, 3.5))
ejes.errorbar(
    comparacion["media"], comparacion["modelo"],
    xerr=comparacion["____"], fmt="o", capsize=4,
)
ejes.set_xlabel("F1-macro (media ± desviación entre pliegues)")
ejes.set_title("Si las barras se solapan, no hay evidencia de diferencia")
plt.tight_layout()
plt.show()

### ✏️ TODO 2 — Poner número a la distinguibilidad

In [ ]:
# TODO 2: ¿qué modelos son distinguibles del mejor?
robustez = optimizacion.____(comparacion)
robustez

In [ ]:
# Autochequeo
indistinguibles = robustez[~robustez["distinguible_del_mejor"]]["modelo"].tolist()
assert len(indistinguibles) >= 2, (
    "revisa: debería haber al menos dos modelos indistinguibles entre sí"
)
print("✅ Modelos que NO se pueden distinguir del mejor:", indistinguibles)
print()
print("   Traducido: con la evidencia que tienes, afirmar que uno es mejor")
print("   que el otro no está respaldado. Cualquiera de los dos es defendible.")

### ✏️ TODO 3 — La consecuencia

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

1. El bosque saca más media que el ensamble. ¿Puedes afirmar que es mejor? ¿Por qué?
2. Si dos modelos son indistinguibles, **¿con qué criterio eliges?**
3. Alguien te muestra una comparación de modelos sin desviaciones, solo medias. ¿Qué le pides?

---
# Bloque 2 · Métricas avanzadas: más allá de un número

El F1 fija implícitamente un umbral de decisión en 0,5. Pero un clasificador no devuelve una
etiqueta: devuelve una **probabilidad**, y el umbral lo eliges tú.

Mover el umbral intercambia precisión por recall. Cuál conviene **lo decide el costo del error
en el dominio**, no la estadística.

### ✏️ TODO 4 — La curva precisión-recall

*(Se usa la curva PR y no la ROC porque con clases desbalanceadas la ROC se ve optimista: su eje
depende de los verdaderos negativos, que aquí son abundantes y fáciles.)*

In [ ]:
# TODO 4: la curva precisión-recall de la clase minoritaria.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import PrecisionRecallDisplay, average_precision_score

prueba = marcada[marcada["particion"] == "prueba"]
X_entrena, y_entrena = entrena[CONFIG["variables"]], entrena[CONFIG["objetivo"]]
X_prueba, y_prueba = prueba[CONFIG["variables"]], prueba[CONFIG["objetivo"]]

modelo = RandomForestClassifier(
    n_estimators=200, max_depth=12, class_weight="balanced",
    random_state=CONFIG["semilla"], n_jobs=-1,
).fit(X_entrena, y_entrena)

positiva = "LEVEL_2"
probabilidades = modelo.____(X_prueba)[:, list(modelo.classes_).index(positiva)]
es_positiva = (y_prueba == positiva).astype(int)

print(f"Average precision (LEVEL_2): {average_precision_score(es_positiva, probabilidades):.4f}")
print(f"Proporción de LEVEL_2       : {es_positiva.mean():.4f}  <- el piso de un clasificador al azar")

PrecisionRecallDisplay.from_predictions(es_positiva, probabilidades, name="bosque aleatorio")
plt.title("Precisión vs recall para la clase minoritaria")
plt.tight_layout()
plt.show()

### ✏️ TODO 5 — Elegir el umbral con criterio de negocio

En la Actividad 2.2 el modelo encontraba el **40 %** de las detecciones difíciles. Veamos qué
umbral haría falta para encontrar el 70 %, y qué se paga por ello.

In [ ]:
# TODO 5: ¿qué umbral hace falta para cada nivel de recall, y qué cuesta?
from sklearn.metrics import precision_recall_curve

precision, recall, umbrales = precision_recall_curve(es_positiva, probabilidades)

objetivos = [0.40, 0.55, 0.70, 0.85]
filas = []
for objetivo in objetivos:
    i = int(np.argmin(np.abs(recall[:-1] - objetivo)))
    predicho = (probabilidades >= umbrales[i]).astype(int)
    filas.append(
        {
            "recall_objetivo": objetivo,
            "umbral": round(float(umbrales[i]), 3),
            "recall_real": round(float(recall[i]), 3),
            "precision": round(float(precision[i]), 3),
            "falsas_alarmas": int(((predicho == 1) & (es_positiva == ____)).sum()),
        }
    )
pd.DataFrame(filas)

**✍️ Tu respuesta al TODO 5:**

*(doble clic aquí y escribe)*

En este dominio, una **falsa alarma** significa que el vehículo desconfía de una detección que
era buena: es prudente de más. Un **falso negativo** significa que confía en una detección mala.

¿Qué umbral elegirías? Justifica con el costo de cada tipo de error, no con la métrica.

---
# Bloque 3 · ⭐⭐ La tabla de selección

Aquí está el **IL 3.4**: *sustentar* la selección. No reportar el máximo, **sustentar**.

Un modelo que gana por un margen indistinguible del ruido, tarda quince veces más y no se puede
explicar **no es la solución óptima**: es la que sacó el número más alto una vez.

### ✏️ TODO 6 — Las cuatro dimensiones juntas

In [ ]:
# TODO 6: métrica, estabilidad, costo e interpretabilidad en una sola tabla.
seleccion = optimizacion.____(comparacion, robustez, CONFIG)
seleccion

### ✏️ TODO 7 — La decisión

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

Elige un modelo y defiéndelo. Tu argumento debe referirse a **las cuatro columnas**, no solo a
la primera.

**Modelo elegido:** `____`

| Criterio | Cómo lo justifica |
|---|---|
| Desempeño | `____` |
| Estabilidad | `____` |
| Costo | `____` |
| Interpretabilidad | `____` |

**Y la pregunta de control:** ¿qué modelo elegirías si el sistema tuviera que responder en
milisegundos dentro del vehículo?

---
# Bloque 4 · Defender la decisión

En el EFT vas a presentar esto y te van a hacer preguntas cruzadas. Estas son las cinco que
más se repiten. Prepara tu respuesta.

### ✏️ TODO 8 — El ensayo

**✍️ Tus respuestas:**

*(doble clic aquí y escribe)*

1. *"¿Por qué este modelo y no el que sacó mejor número?"* → `____`
2. *"¿Cómo sabes que no está sobreajustado?"* → `____`
3. *"Si te doy el doble de datos, ¿mejoraría?"* → `____`
4. *"¿Qué pasa si los datos de producción no se parecen a los de entrenamiento?"* → `____`
5. *"¿Cuánto de tu mejora es real y cuánto es azar?"* → `____`

---
# Cierre · Informe de selección

Este informe **es** el apartado de modelamiento del EFT. Guárdalo.

### El problema

**Qué se predice:** `____` · **Métrica principal y por qué:** `____`
**Costo del error** (qué pasa con un falso positivo y con un falso negativo): `____`

### Esquema de validación

| Campo | Valor |
|---|---|
| Tipo | `____` |
| Pliegues | `____` |
| Agrupación | `____` |
| Segmentos compartidos | `____` |
| ¿Se usó la prueba para elegir algo? | `____` |

### Comparación cuantitativa

| Modelo | F1-macro | Desv. | ¿Distinguible del mejor? | Segundos | Interpretabilidad |
|---|---|---|---|---|---|
| Baseline | | | | | |
| `____` | | | | | |
| `____` | | | | | |

**Modelos indistinguibles entre sí:** `____`

### La decisión

**Modelo elegido:** `____`

| Criterio | Justificación |
|---|---|
| Desempeño | `____` |
| Estabilidad | `____` |
| Costo | `____` |
| Interpretabilidad | `____` |

**Umbral de decisión elegido y por qué:** `____`

### Límites, dichos por mí antes de que me los pregunten

**Qué mejora sería real y cuál sería ruido:** `____`
**En qué condiciones NO debe usarse este modelo:** `____`
**Qué haría falta para la próxima mejora:** `____`

> Esa última sección es la que separa un informe técnico de un informe de ventas. Un modelo
> presentado con sus límites es utilizable; uno presentado como universal, no.